In [4]:
# Se estiver rodando no Google Colab, esta célula abre a janela de upload
# e organiza o arquivo na subpasta dataset/, igual à estrutura do repositório.
# Selecione o arquivo do miniprojeto: Base Varejo.csv

import os

try:
    from google.colab import files
    enviados = files.upload()
    os.makedirs("dataset", exist_ok=True)
    for nome in enviados:
        os.replace(nome, f"dataset/{nome}")
except ImportError:
    print("Rodando localmente (VS Code) — o arquivo já está em dataset/, na pasta deste mini-projeto.")

Rodando localmente (VS Code) — o arquivo já está em dataset/, na pasta deste mini-projeto.


1 - Carregar a base Varejo.csv com pandas e mostrar: número de registros, colunas e tipos de dados.

In [ ]:
import csv
import pandas as pd
from IPython.display import display

with open("dataset/Base Varejo.csv", "r", encoding="utf-8") as arquivo:
    base_varejoCsv = csv.DictReader(arquivo, delimiter=";")

    base_varejo = pd.DataFrame(base_varejoCsv)

# Número de registros (linhas)
print("Número de registros:", base_varejo.shape[0])

# Número de colunas
print("Número de colunas:", base_varejo.shape[1])

# Tipos de dados de cada coluna
print("\nTipos de dados:")
print(base_varejo.dtypes)

Número de registros: 830000
Número de colunas: 11

Tipos de dados:
DATA         str
CO_ID        str
CL_ID        str
CL_GENERO    str
CL_EC        str
CL_FHL       str
CL_SEG       str
PR_ID        str
PR_CAT       str
PR_NOME      str
             str
dtype: object


2 - Verificar e reportar ao menos dois problemas básicos: valores nulos por coluna, duplicatas e possíveis inconsistências (ex.: datas inválidas ou categorias vazias).

Funções para tranformar colunas do dataframe em Strings(limpar_texto), Integer(limpar_inteiro) e Float(limpar_decimal) e Datetime(limpar_data)

In [14]:
import re
import pandas as pd
import numpy as np


def limpar_texto(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()
    valor = re.sub(r"\s+", " ", valor)        # remove espaços duplicados
    valor = re.sub(r"[^\w\sÀ-ÿ-]", "", valor) # remove caracteres indesejados

    return valor


def limpar_inteiro(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    # mantém apenas números e sinal negativo
    valor = re.sub(r"[^\d-]", "", valor)

    if valor in ("", "-"):
        return np.nan

    return int(valor)


def limpar_decimal(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip()

    # Exemplo: "R$ 1.234,56" -> "1234.56"
    valor = re.sub(r"[^\d,.-]", "", valor)
    valor = valor.replace(".", "")
    valor = valor.replace(",", ".")

    if valor in ("", "-", ".", "-."):
        return np.nan

    return float(valor)


def limpar_data(valor):
    if pd.isna(valor):
        return pd.NaT

    return pd.to_datetime(
        valor,
        dayfirst=True,
        errors="coerce"
    )

In [ ]:
# DATA: Data da compra;
base_varejo["DATA"] = base_varejo["DATA"].apply(limpar_data)
# CO_ID: Identificação do número de compra (número da nota fiscal);
base_varejo["CO_ID"] = base_varejo["CO_ID"].apply(limpar_inteiro)

# CL_ID: Identificação do cliente (número do cliente);
base_varejo["CL_ID"] = base_varejo["CL_ID"].apply(limpar_inteiro)
# CL_GENERO: Sexo biológico informado pelo cliente;
base_varejo["CL_GENERO"] = base_varejo["CL_GENERO"].apply(limpar_texto)
# CL_EC: Estado civil do cliente: 1 - Casado ou união estával; 2 - Divorciado; 3 - Separado; 4 - Solteiro; 5 - Viúvo.
base_varejo["CL_EC"] = base_varejo["CL_EC"].apply(limpar_inteiro)
# CL_FHL: Número de filhos do cliente;
base_varejo["CL_FHL"] = base_varejo["CL_FHL"].apply(limpar_inteiro)
# CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
base_varejo["CL_SEG"] = base_varejo["CL_SEG"].apply(limpar_texto)

# PR_ID: Código do produto (SKU) adquirido;
base_varejo["PR_ID"] = base_varejo["PR_ID"].apply(limpar_inteiro)
# PR_CAT: Categoria do produto adquirido;
base_varejo["PR_CAT"] = base_varejo["PR_CAT"].apply(limpar_texto)
# PR_NOME: Nome do produto adquirido.
base_varejo["PR_NOME"] = base_varejo["PR_NOME"].apply(limpar_texto)

# Verificar os tipos após transformação
print(base_varejo.dtypes)

DATA         datetime64[us]
CO_ID                 int64
CL_ID                 int64
CL_GENERO               str
CL_EC                 int64
CL_FHL                int64
CL_SEG                  str
PR_ID                 int64
PR_CAT                  str
PR_NOME                 str
                        str
dtype: object


3 - Fazer as três etapas de limpeza mínima necessária: remover ou imputar nulos (explique a escolha), eliminar duplicatas relevantes e ajustar tipos de dados (ex.: converter coluna DATA para datetime).

In [21]:
import pandas as pd
import numpy as np

# Identificar valores nulos
print(base_varejo.isnull().sum())

# Identificar registros duplicados
print("Duplicatas:", base_varejo.duplicated().sum())

# Remover duplicatas
base_varejo = base_varejo.drop_duplicates()

# Verificar resultado
print(base_varejo.info())


DATA              0
CO_ID             0
CL_ID             0
CL_GENERO         0
CL_EC             0
CL_FHL            0
CL_SEG            0
PR_ID             0
PR_CAT            0
PR_NOME           0
             830000
dtype: int64
Duplicatas: 96553
<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 11 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   DATA       733447 non-null  datetime64[us]
 1   CO_ID      733447 non-null  int64         
 2   CL_ID      733447 non-null  int64         
 3   CL_GENERO  733447 non-null  str           
 4   CL_EC      733447 non-null  int64         
 5   CL_FHL     733447 non-null  int64         
 6   CL_SEG     733447 non-null  str           
 7   PR_ID      733447 non-null  int64         
 8   PR_CAT     733447 non-null  str           
 9   PR_NOME    733447 non-null  str           
 10             0 non-null       str           
dtypes: datetime64[us](1), int